In [1]:
import numpy as np
import h5py
import pandas as pd
import skimage as ski
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from pathlib import Path
from stardist.models import StarDist2D,Config2D
from csbdeep.utils import normalize
from stardist.plot import render_label
import tifffile
import sys
import shutil
import preprocess_for_training as prep

In [2]:
Data=Path("../Data")
training_images = Path("../Data/Training Pool/Images")
labels = Path("../Data/Training Pool/Labels")

#make dictionary with stacks in training pool
training={}
for img in training_images.glob("*.tif"):
    name = img.stem.rsplit("_", 1)[0] #has names likes FS_250903_007_0_labels so split from right, take everything before last underscore
    # Find matching label file
    label = labels / f"{name}_labels.tif" 
    
    if label.exists():
        training[name] = {
            "image": ski.io.imread(img),
            "label": ski.io.imread(label)}
#also load h5 files to get the magnification for these names
h5_files = {h5_file.stem: h5_file for h5_file in Data.rglob("*.h5")} #look for files with .h5 extension

for name, data in training.items(): 
    h5_target_stem = name.rsplit("_", 1)[0] 
    path=h5_files.get(h5_target_stem)
    #get magnification from the h5 file
    with h5py.File(path, 'r') as f:
        attrs = f['data'].attrs
        x_ampli = attrs['Scanner.X_Amplitude'] #should be same as y_ampli
        x_points = attrs['Scanner.X_Points']
        calibration = attrs['Scanner.X_Calibration']
        mag = (x_ampli * calibration) / x_points

        #add magnification to dictionary
        data["magnification"]=mag 

In [3]:
#take the images/labels in training pool and process them using preprocess_for_training
#rescale them all consistently (and use the same TARGET SIZE when preprocessing for inference!)
TARGET_SIZE=20

for data in training.values():
    rescaled_img, rescaled_labels = prep.prepare_image_and_label(
        data["image"], 
        data["label"], 
        data["magnification"], 
        TARGET_SIZE
    )
    #write updated img/labels in same dictionary
    data["image"] = rescaled_img
    data["label"] = rescaled_labels

In [ ]:
plt.imshow()

In [7]:
#check that none are too small for patch size (must include full spot and context)
too_small = [name for name, data in training.items()
             if data["image"].shape[0] < 96 or data["image"].shape[1] < 96]
print(too_small)

[]


In [15]:
X = []  # Images
Y = []  # Instance Label Masks
for data in training.values():
    X.append(data["image"])
    Y.append(data["label"])

#split into train and validation 
split_idx = int(0.8 * len(X))
X_train, Y_train = X[:split_idx], Y[:split_idx]
X_val, Y_val = X[split_idx:], Y[split_idx:]

In [17]:
def augment_func(x, y):
    """
    Applies joint geometric and intensity transformations 
    to a single image (x) and label mask (y).
    """
    # 1. Random Horizontal Flip
    if np.random.rand() > 0.5:
        x = np.fliplr(x)
        y = np.fliplr(y)
        
    # 2. Random Vertical Flip
    if np.random.rand() > 0.5:
        x = np.flipud(x)
        y = np.flipud(y)
        
    # 3. Random 90-degree Rotations (0, 90, 180, or 270 degrees)
    k = np.random.randint(0, 4)
    x = np.rot90(x, k)
    y = np.rot90(y, k)
    
    # 4. Mild Intensity Jitter (multiply brightness)
    scale = np.random.uniform(0.9, 1.1)
    x = x * scale
    
    return x, y

In [29]:
#load pretrained model and decide where updated one will be saved
pretrained = StarDist2D.from_pretrained('2D_versatile_fluo')
#modify configuration
conf = pretrained.config
conf.train_patch_size = (96, 96)
conf.train_batch_size = 8

name='v1_finetuning'
basedir='./models'
my_model = StarDist2D(conf, name=name, basedir=basedir)

#transfer pretrained weights
my_model.keras_model.set_weights(pretrained.keras_model.get_weights())

#NOTE - to call model later (eg from another notebook) use
#same_model = StarDist2D(None, name='v1_finetuning', basedir='models')

Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.
Using default values: prob_thresh=0.5, nms_thresh=0.4.


base_model.py (203): output path for model already exists, files may be overwritten: /Users/iris/Documents/TUM_project/porphyrins/project/models/v1_finetuning


In [ ]:
#training
my_model.train(X_train, Y_train, validation_data=(X_val, Y_val), epochs=100, steps_per_epoch=len(X_train),augmenter=augment_func)

In [ ]:
my_model.optimize_thresholds(X_val, Y_val)

In [ ]:
#To see how model has performed on training data 
idx = 2
img_val = X_train[idx]
gt_val = Y_train[idx] #ground truth 

# Predict instances using the optimized thresholds automatically
labels_pred, details = my_model.predict_instances(img_val)

# Plot Ground Truth vs Prediction
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(img_val, cmap='gray')
ax[0].set_title("Input Image")

ax[1].imshow(render_label(gt_val, img=img_val, alpha=0.5, cmap='prism'))
ax[1].set_title("Ground Truth Masks")
ax[1].axis('off')

ax[2].imshow(render_label(labels_pred, img=img_val, alpha=0.5, cmap='prism'))
ax[2].set_title("StarDist Predictions")
ax[2].axis('off')

plt.tight_layout()
plt.show()